In [1]:
import pandas as pd

In [2]:
%run ../gs_slimming.py
print("="*120)
print("gs_slimming done.")
print("="*120)
%run flattening.py
print("="*120)
print("flattening done.")
print("="*120)

Mismatches: {'ViacomCBS_ESG Report_2020-2021_vFINAL.pdf'}

No. of gold_standard reports: 139

No. of downloadable reports: 113

No. of reports in gs_slim: 54


status
partial     45
complete     9
Name: count, dtype: int64

scopes_present
[1, 2lb]            18
[1, 2lb, 3]         15
[1, 2lb, 2mb, 3]     9
[1, 2lb, 2mb]        6
[1, 2mb, 3]          1
[1, 3]               1
[1]                  1
[2lb, 3]             1
b]                1
[3]                  1
Name: count, dtype: int64
gs_slimming done.

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv
  Reports    : 54
  Zeilen     : 672
  Fehler     : 0
  Gespeichert: /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv

  Flattening 54 JSONs => /Users/jannikkuhlmann/VSC/LaTeX/2026_BA_Code/evaluations/PipelineB/PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv
  Reports    : 5

## Import slimmed down Gold-Standard

In [3]:
gs = pd.read_json("../gs_slim.json")
gs["year"] = gs["year"].astype(str) #Correction needed for e.g. fiscal years "FY 2021/2022"

## Import all 3x2=6 Extractions

In [15]:
think1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv")
think2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Thinking.csv")

moe1   = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv")
moe2   = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-30B-A3B-Thinking.csv")

instr1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv")
instr2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Instruct.csv")

df_to_merge = [
    (think1, "_t1"),
    (think2, "_t2"),
    (moe1, "_m1"),
    (moe2, "_m2"),
    (instr1, "_i1"),
    (instr2, "_i2")
]

## Matching Extractions to Gold-Standard Scheme

In [25]:
# Zeigt alle einzigartigen "scopes" aus den JSON-Dateien unter PipelineB-Answers
from pathlib import Path
import json
from collections import Counter, defaultdict

input_dir = Path("./PipelineB-Answers")
input_folder  = sorted(p for p in input_dir.iterdir() if p.is_dir())

for input_root in input_folder :

    # input_root = Path("evaluations/PipelineB/PipelineB-Answers")
    files = sorted(input_root.rglob("*.json"))
    print(f"Found {len(files)} JSON files under {input_root}\n")

    scopes_set = set()
    scope_counts = Counter()
    per_file = defaultdict(list)

    for f in files:
        try:
            data = json.loads(f.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"Failed to load {f}: {e}")
            continue

        emissions = data.get("emissions", {})
        if not isinstance(emissions, dict):
            print(f"{f}: 'emissions' is not a dict (type={type(emissions)})")
            continue

        for scope in emissions.keys():
            scopes_set.add(scope)
            scope_counts[scope] += 1
            # Try to store a path relative to cwd; if that fails (ValueError), store the full path
            try:
                rel = f.relative_to(Path.cwd())
            except Exception:
                rel = f
            per_file[scope].append(rel)

    # Ausgabe der unique Scopes
    print("Unique scopes (sorted):")
    for s in sorted(scopes_set):
        print(f" - {s}  ({scope_counts[s]} files)")

    # Hilfsansichten, um Tippfehler zu finden
    print("\nScopes containing '-' (possible typos vs underscores):")
    for s in sorted(scopes_set):
        if '-' in s:
            print(f" - {s}  ({scope_counts[s]} files)")
            for p in per_file[s][:10]:
                print("    ", p)

Found 54 JSON files under PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking

Unique scopes (sorted):
 - scope_1  (54 files)
 - scope_2_location_based  (47 files)
 - scope_2_market_based  (54 files)
 - scope_3  (51 files)

Scopes containing '-' (possible typos vs underscores):
Found 54 JSON files under PipelineB-Answers/1st_Qwen3-VL-32B-Instruct

Unique scopes (sorted):
 - scope_1  (54 files)
 - scope_2_location_based  (54 files)
 - scope_2_market_based  (54 files)
 - scope_3  (54 files)

Scopes containing '-' (possible typos vs underscores):
Found 54 JSON files under PipelineB-Answers/1st_Qwen3-VL-32B-Thinking

Unique scopes (sorted):
 - scope_1  (54 files)
 - scope_2_location_based  (54 files)
 - scope_2_market_based  (53 files)
 - scope_3  (53 files)

Scopes containing '-' (possible typos vs underscores):
Found 54 JSON files under PipelineB-Answers/2nd_Qwen3-VL-30B-A3B-Thinking

Unique scopes (sorted):
 - scope_1  (54 files)
 - scope_2_location_based  (49 files)
 - scope_2_market_based

In [19]:
for df, sx in df_to_merge :
    print(f"{sx} : {df["scope"].unique().tolist()}")

_t1 : ['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']
_t2 : ['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']
_m1 : ['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']
_m2 : ['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']
_i1 : ['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']
_i2 : ['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']


In [5]:
scope_mapping = {
    "scope_1": "1",
    "scope_2_location_based": "2lb",
    "scope_2_market_based" : "2mb",
    "scope_3" : "3"
}

# Prints out only if sth. goes wrong
for df, sf in df_to_merge:
    # Replace scope definitons to match Gold-Standard
    df['scope'] = df['scope'].replace(scope_mapping)
    
    # Ensure every value column is a float64 to match Gold-Standard
    converted = pd.to_numeric(
        df['value'].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )
    failed = df['value'][converted.isna() & df['value'].notna()]
    if len(failed) > 0:
        print(f"{sf}: {len(failed)} nicht konvertierbar: {failed.unique()}")
    df['value'] = converted


Checking if all `report_names` are the same

And checking if all `report_names` from the extractions are exactly matched in the GoldStandard

In [6]:
print("Think ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Moe   ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print("Instr ok:", all(think1["report_name"].unique() == think2["report_name"].unique()))
print()

# print(think1["report_name"].isin(gs["report_name"]).all())
# print(gs["report_name"].isin(think1["report_name"]).all())

in_think1_not_gs  = set(think1["report_name"]) - set(gs["report_name"])
in_gs_not_think1  = set(gs["report_name"])     - set(think1["report_name"])

print(f"In think1, nicht in GS: {len(in_think1_not_gs)} ==> {"OK" if len(in_think1_not_gs) == 0 else "NOK"}")
for r in sorted(in_think1_not_gs): print(" ", r)

print(f"\nIn GS, nicht in think1: {len(in_gs_not_think1)} ==> {"OK" if len(in_gs_not_think1) == 0 else "NOK"}")
for r in sorted(in_gs_not_think1): print(" ", r)

Think ok: True
Moe   ok: True
Instr ok: True

In think1, nicht in GS: 0 ==> OK

In GS, nicht in think1: 0 ==> OK


## Merging Extraction Values and Gold-Standard

In [7]:
merge_on = ["report_name", "scope", "year"]
agg_cols = ["value", "unit", "label"]

In [8]:
for df, sf in df_to_merge:
    dups = df.duplicated(subset=merge_on).sum()
    print(f"{sf}: {dups} doppelte Keys")


_t1: 147 doppelte Keys
_t2: 160 doppelte Keys
_m1: 202 doppelte Keys
_m2: 148 doppelte Keys
_i1: 190 doppelte Keys
_i2: 300 doppelte Keys


=> That's way I do not need a normal merge, but consolidate every extracted value, unit, and label from every dataframe (think1, think2, moe1, ...) value into a list.

Then I can validate if the gs-value is extracted or not.

These are represented as "objects" when calling merged.info() but are just python lists.

In [9]:
merged = gs.copy() # Setting a starting point for the loop. Everything has to be mapped to the Gold-Standard

for df, sf in df_to_merge:
    agg = (
        df.groupby(merge_on)[agg_cols]
        .agg(list)
        .reset_index()
        .rename(columns={col: f"{col}{sf}" for col in agg_cols})
    )
    merged = pd.merge(merged, agg, on=merge_on, how="left")
    
merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   year             2208 non-null   str    
 2   scope            2208 non-null   str    
 3   page             490 non-null    str    
 4   value            489 non-null    float64
 5   unit             488 non-null    str    
 6   unit_normalized  488 non-null    str    
 7   label            489 non-null    str    
 8   status           2208 non-null   str    
 9   scopes_present   2208 non-null   object 
 10  years_present    2208 non-null   object 
 11  value_t1         497 non-null    object 
 12  unit_t1          497 non-null    object 
 13  label_t1         497 non-null    object 
 14  value_t2         506 non-null    object 
 15  unit_t2          506 non-null    object 
 16  label_t2         506 non-null    object 
 17  value_m1         468 non-

### Rearranging Columns

In [10]:
gs_cols = [
    'report_name', 'status', 'scopes_present', 'years_present', # Per-Report granularity
    'year', 'scope',                                            # Per category
    'page', 'value', 'unit',                                    # About the value as-is in the report
    'unit_normalized', 'label',                                 # Additional information
]

pipeline_cols = (
    [f"value{s}" for _, s in df_to_merge] +
    [f"unit{s}"  for _, s in df_to_merge] +
    [f"label{s}" for _, s in df_to_merge]
)

merged = merged[gs_cols + pipeline_cols]

merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 2208 entries, 0 to 2207
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   report_name      2208 non-null   str    
 1   status           2208 non-null   str    
 2   scopes_present   2208 non-null   object 
 3   years_present    2208 non-null   object 
 4   year             2208 non-null   str    
 5   scope            2208 non-null   str    
 6   page             490 non-null    str    
 7   value            489 non-null    float64
 8   unit             488 non-null    str    
 9   unit_normalized  488 non-null    str    
 10  label            489 non-null    str    
 11  value_t1         497 non-null    object 
 12  value_t2         506 non-null    object 
 13  value_m1         468 non-null    object 
 14  value_m2         464 non-null    object 
 15  value_i1         490 non-null    object 
 16  value_i2         496 non-null    object 
 17  unit_t1          497 non-

## Saving created DataFrame

In [11]:
merged.to_csv("gs_extractions_raw.csv", index=False)
merged.to_json("gs_extractions_raw.json", index=False, orient="records", indent=4)

## To later read back the files:
# pd.read_csv("gs_extractions_raw.csv")   # Lists as Strings. Needs ast.literal_eval
# pd.read_json("gs_extractions_raw.json", orient="records")  # Lists instant usable


## Creating Overview which scopes are present for which years of a given report

In [12]:
year_scope_df = (
    gs[gs["value"].notna()]
    .groupby(["report_name", "year"])["scope"]
    .apply(lambda x: sorted(x.unique()))
    .reset_index()
    .rename(columns={"scope": "scopes"})
)

year_scope_df["_sort"] = year_scope_df["report_name"].str.lower() # need to circumvent ASCII case-sensitive sorting
year_scope_df = year_scope_df.sort_values(["_sort", "year"], ignore_index=True).drop(columns="_sort")

year_scope_df.to_csv("year_scope_df.csv", index=False)
year_scope_df.to_json("year_scope_df.json", index=False, orient="records", indent=4)
year_scope_df.head()


,report_name,year,scopes
0,acuity brands inc_2022_report,2022,"[1, 2lb, 2mb, 3]"
1,addtech_2022_report,2019,"[1, 2lb, 3]"
2,addtech_2022_report,2020,"[1, 2lb, 2mb, 3]"
3,addtech_2022_report,2021,"[1, 2lb, 2mb, 3]"
4,aixtron_2020_report,2019,"[1, 2lb, 3]"


In [13]:
year_scope_df[year_scope_df["report_name"] == "acuity brands inc_2022_report"]["scopes"]

0    [1, 2lb, 2mb, 3]
Name: scopes, dtype: object